# FİNAL: 3 seed eğitimi + mühürlü test (tek seferde)

Seçilen yapılandırma: BERTurk + augmented-v3 + kural katmanı + son işlem.
**Çalışma zamanı → T4 GPU → Tümünü çalıştır.** Yaklaşık 3 saat sürer; başında beklemeye gerek yok.
Her seed bitince model Drive'a kaydedilir. Bağlantı koparsa yeniden **Tümünü çalıştır**: hazır modeller atlanır,
kaldığı yerden devam eder. Mühürlü test yalnızca üç model de hazır olduğunda, bir kez çalışır.

In [ ]:
import os, pathlib, shutil, subprocess, zipfile, json, torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı türünü T4 GPU yap."
from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/turkce-llm-dlp')
os.chdir('/content')
shutil.rmtree('/content/proje', ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/irmakyaprakk/turkce-llm-dlp.git', '/content/proje'], check=True)
for src in sorted({i.parent for i in pathlib.Path('/content/drive/MyDrive').rglob('BUNDLE_INFO.txt')}
                  | set(pathlib.Path('/content/drive/MyDrive').rglob('turkce-dlp-colab-bundle*.zip')), key=lambda p: p.name):
    tmp = pathlib.Path('/content/_paket'); shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src, tmp) if src.is_dir() else zipfile.ZipFile(src).extractall(tmp)
    root = next((p.parent for p in tmp.rglob('BUNDLE_INFO.txt')), tmp)
    if (root / 'data').exists():
        shutil.copytree(root / 'data', '/content/proje/data', dirs_exist_ok=True)
os.chdir('/content/proje')
assert pathlib.Path('data/ner/processed/augmented-v3/train.jsonl').exists(), 'augmented-v3 verisi yok (v5 paketi Drive\'da olmalı)'
!pip -q install -r training/ner/requirements.txt
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# Eksik seed'leri eğit. Drive'da hazır olanlar atlanır.
for seed in (42, 43, 44):
    saved = DRIVE / 'runs' / f'augmented_v3_seed{seed}'
    legacy = DRIVE / 'runs' / 'augmented_v3'  # ilk v3 koşusu seed 42 ile bu adla kaydedildi
    if seed == 42 and not (saved / 'model').exists() and (legacy / 'model').exists():
        shutil.copytree(legacy, saved, dirs_exist_ok=True)
    if (saved / 'model').exists():
        print(f'seed {seed}: hazır, atlanıyor'); continue
    out = f'/content/runs/augmented_v3_seed{seed}'
    print(f'seed {seed}: eğitim başlıyor')
    !USE_TF=0 python training/ner/train.py --data-dir data/ner/processed/augmented-v3 --labels-path training/ner/labels.json --output-dir {out} --epochs 3 --seed {seed} --fp16 > /content/train-{seed}.log 2>&1
    !tail -3 /content/train-{seed}.log
    shutil.rmtree(saved, ignore_errors=True)
    shutil.copytree(out, saved, ignore=shutil.ignore_patterns('checkpoints'))
    print(f'seed {seed}: Drive\'a kaydedildi')

In [ ]:
# Mühürlü final test (bir kez) + geliştirme seti, üç seed, ortalama ± std
!python training/ner/build_gold_set.py --sealed --out-dir data/ner/processed/final-test > /dev/null
!python training/ner/build_gold_set.py --out-dir data/ner/processed/realistic-v2 > /dev/null
results = {'final': [], 'gelistirme': []}
for seed in (42, 43, 44):
    model = DRIVE / 'runs' / f'augmented_v3_seed{seed}' / 'model'
    for name, data, splits in (('final', 'data/ner/processed/final-test', ['test']),
                               ('gelistirme', 'data/ner/processed/realistic-v2', ['dev', 'test'])):
        preds = f'/content/{name}-{seed}.jsonl'
        open(preds, 'w').close()
        for split in splits:
            !USE_TF=0 python training/ner/evaluate_model.py --model-dir "{model}" --data-dir {data} --split {split} --dump-predictions /content/p.jsonl > /dev/null
            !cat /content/p.jsonl >> {preds}
        target = DRIVE / 'runs' / f'augmented_v3_seed{seed}' / f'{name}-pipeline.json'
        !python training/ner/evaluate_rules.py --data-dir {data} --splits {' '.join(splits)} --with-predictions {preds} --output-json "{target}" > /dev/null
        results[name].append(str(target))
summary = {}
for name, files in results.items():
    out = subprocess.run(['python', 'training/ner/aggregate_seeds.py', *files], capture_output=True, text=True, check=True).stdout
    (DRIVE / 'runs' / f'{name}-summary.json').write_text(out)
    metrics = json.loads(out)['metrics']
    summary[name] = {k.replace('pipeline_model_plus_rules', 'hibrit').replace('.metrics', '').replace('.token_coverage', ''):
                     f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in metrics.items()
                     if k.startswith('pipeline_model_plus_rules') and any(x in k for x in ('_f1', 'overall_precision', 'overall_coverage', 'person_coverage', 'org_coverage', 'address_coverage', 'location_coverage'))}
print(json.dumps(summary, ensure_ascii=False, indent=1))